# Testing Models & Data Quality

> 📘 **Python Mastery** · Module 18 — MLOps · Lesson 5/6

Untested code embarrasses you in code review; untested DATA quietly poisons everything downstream until someone asks why recall fell off a cliff. This lesson adds the missing layers to your test suite: schema and expectation checks on every batch, metric gates that block weak models, and behavioural contracts that pin down what a model may and may not do.

## 🎯 Learning Objectives

- Sort every test you own into three layers: unit (code), data (inputs), behaviour (model)
- Write pytest test functions and run a suite headlessly from Python with `-m pytest`
- Validate incoming batches against a declared schema before training or serving touches them
- Encode statistical expectations: null shares, value ranges, category sets, ID uniqueness
- Gate deployment on metric thresholds and behavioural properties (direction, invariance)
- Judge when hand-rolled checks suffice and when a tool like Great Expectations earns its keep

## 1. Three Layers of Tests

Most teams test layer 1 and improvise the rest — which is how a renamed column reaches
production. Name the layers, then staff all three:

| Layer | Target | Example | Catches |
|---|---|---|---|
| **Unit** | your code | `add_vat(100) == 118` | logic bugs, regressions |
| **Data** | each incoming batch | "no null amounts, ages 18–100" | upstream schema drift, corrupt feeds |
| **Behaviour** | the fitted model | "more hours never lowers the score" | silent training bugs, unfair flips |

Layer 2 runs at ingestion time — *before* the batch lands anywhere near `fit()`.
Layer 3 runs before promotion — a model that fails behaviour does not deploy, however
pretty its holdout accuracy looks.

## 2. pytest in Five Minutes

pytest discovers any `test_*.py`, collects every `test_*` function, and treats a bare
`assert` as a rich failure report. Parametrising one function multiplies it into many cases.

**Workflow:**

```bash
pip install pytest          # once
pytest tests/ -q            # quiet, fast, exit code 0 only if EVERYTHING passed
```

CI reads that exit code — which is exactly how Lesson 06 will block releases. You can run
pytest programmatically too, so notebooks and pipelines can invoke suites without a shell:

In [ ]:
# Write a tiny suite to disk, run it with pytest, read the verdict
import subprocess
import sys
from pathlib import Path

test_file = Path("sample_data/test_pricing.py")
test_file.parent.mkdir(exist_ok=True)
test_file.write_text('''
from pricing import add_vat

def test_add_vat_basic():
    assert add_vat(100.0) == 118.0

def test_add_vat_zero():
    assert add_vat(0.0) == 0.0
''')
(Path("sample_data") / "pricing.py").write_text(
    "TAX = 0.18\n\n\ndef add_vat(amount: float) -> float:\n"
    "    return round(amount * (1 + TAX), 2)\n")

result = subprocess.run(
    [sys.executable, "-m", "pytest", str(test_file), "-q", "--no-header"],
    capture_output=True, text=True, cwd=str(test_file.parent),
)
print(result.stdout.strip().splitlines()[-1])     # '2 passed in 0.0xs'
assert result.returncode == 0                     # <- what CI would check

## 3. Data Validation Gates

Declare the contract a batch must honour, check it mechanically, refuse the batch otherwise.
A validator is deliberately boring: columns present, types right, nulls rare, values sane,
categories known. Boring is the point — it runs every hour, forever, without judgement.

**Syntax:** one function, one verdict:

```python
violations = validate_batch(df, rules)
if violations:
    raise DataQualityError(violations)   # stop the pipeline HERE
```

In [ ]:
# The gate: clean batch sails through, corrupted batch is stopped cold
import pandas as pd

RULES = {
    "columns": ["student_id", "hours_studied", "city"],
    "null_share_max": {"hours_studied": 0.05},
    "range": {"hours_studied": (0, 80)},
    "allowed": {"city": {"Dhaka", "Chattogram", "Sylhet"}},
}


def validate_batch(df: pd.DataFrame, rules: dict) -> list[str]:
    problems = []
    missing = set(rules["columns"]) - set(df.columns)
    if missing:
        problems.append(f"missing columns: {sorted(missing)}")
    for col, limit in rules["null_share_max"].items():
        if col in df and df[col].isna().mean() > limit:
            problems.append(f"{col}: null share {df[col].isna().mean():.2%} > {limit:.0%}")
    for col, (lo, hi) in rules["range"].items():
        if col in df and ((df[col] < lo) | (df[col] > hi)).any():
            bad = df.loc[(df[col] < lo) | (df[col] > hi), col].iloc[0]
            problems.append(f"{col}: value {bad} outside [{lo}, {hi}]")
    for col, allowed in rules["allowed"].items():
        if col in df:
            strangers = set(df[col].dropna()) - allowed
            if strangers:
                problems.append(f"{col}: unexpected values {sorted(strangers)}")
    return problems


clean = pd.DataFrame({"student_id": [1, 2, 3],
                      "hours_studied": [4.0, 7.5, 2.0],
                      "city": ["Dhaka", "Sylhet", "Chattogram"]})
corrupt = pd.DataFrame({"student_id": [1, 2, 3],
                        "hours_studied": [4.0, None, 95.0],     # null + impossible value
                        "city": ["Dhaka", "Paris", "Chattogram"]})  # unknown city

print("clean batch  :", validate_batch(clean, RULES) or "OK - proceed to training")
print("corrupt batch:", *validate_batch(corrupt, RULES), sep="\n  ")

## 4. Statistical Expectations

Schema checks catch *structure*; expectations catch *statistics*. A column can exist, be
perfectly typed, and still be wrong: every `student_id` duplicated, mean spend suddenly 40x
its usual band, a churn label that is somehow always `1`. Expectations encode yesterday's
knowledge as today's assertions.

| Expectation | Encodes | Fires when |
|---|---|---|
| ID uniqueness | one row per entity | duplicate ingestion |
| Row-count floor | feed completeness | upstream job silently died |
| Mean within band | stable population | distribution shift (drill-down: Lesson 06 PSI) |
| Label balance | realistic classes | join bug collapsed labels to one value |

In [ ]:
# Expectations on the batch: cheap arithmetic, expensive bug caught early
import numpy as np
import pandas as pd

batch = pd.DataFrame({
    "student_id": [101, 102, 103, 104],
    "hours_studied": [4.0, 7.5, 2.0, 6.5],
    "label": [0, 1, 0, 1],
})


def check_expectations(df: pd.DataFrame) -> list[str]:
    failures = []
    if df["student_id"].duplicated().any():
        failures.append("student_id: duplicates present")
    if len(df) < 4:
        failures.append(f"row count {len(df)} below floor of 4")
    mean_hours = df["hours_studied"].mean()
    if not 0 <= mean_hours <= 20:
        failures.append(f"mean hours {mean_hours:.1f} outside expected band 0-20")
    if set(df["label"].unique()) - {0, 1}:
        failures.append("label: values outside {0, 1}")
    return failures


verdict = check_expectations(batch)
print("expectation failures:", verdict or "none - batch accepted")

tampered = batch.assign(hours_studied=[400.0, 380.0, 410.0, 395.0])  # units glitch
print("after unit glitch  :", check_expectations(tampered))

## 5. Model Behaviour Tests

Offline metrics summarise a model in one number; behaviour tests pin down *shape*. Two
workhorse properties:

- **Directionality** — a feature that should move the prediction up must never move it down.
- **Invariance** — a feature the model was never given must not change its answer.

Add a **metric gate**: promotion requires beating an explicit threshold. Together they turn
"the model feels fine" into assertions a machine can enforce.

In [ ]:
# Direction + invariance + a metric gate, enforced mechanically
import numpy as np
from sklearn.linear_model import LinearRegression

# tiny study model: score grows with hours; favourite colour is NOT a feature
hours = np.array([[1], [2], [3], [4], [5], [6]])
scores = np.array([40, 47, 55, 60, 70, 76])
model = LinearRegression().fit(hours, scores)


def test_directionality():
    preds = model.predict(np.array([[2.0], [4.0], [6.0]]))
    assert preds[0] < preds[1] < preds[2], "more hours lowered the score!"


def test_invariance_to_irrelevant_input():
    # the API might receive extra fields; the MODEL must ignore them
    base = float(model.predict([[3.0]])[0])
    with_extra = float(model.predict([[3.0]])[0])   # colour never enters X
    assert abs(base - with_extra) < 1e-12


def test_metric_gate():
    r_squared = model.score(hours, scores)
    assert r_squared >= 0.90, f"R^2 {r_squared:.3f} below the 0.90 promotion bar"


for test_fn in (test_directionality, test_invariance_to_irrelevant_input, test_metric_gate):
    test_fn()
    print(f"{test_fn.__name__}: PASS")

## 6. When to Reach for Great Expectations

Hand-rolled validators are perfect while your rules fit in one screen. A framework buys you
a catalogue of pre-written expectations, HTML data-docs reports and integration with
orchestration — worth it when many teams share many tables:

| You are here | Reach for |
|---|---|
| One project, a dozen rules, one team | hand-rolled `validate_batch` (this lesson) |
| Many tables, many owners, audits demanded | Great Expectations / Soda / dbt tests |

Whatever the tool, the *contract* stays yours: nobody can declare your business rules for you.

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Exact float comparisons in tests | `(a + b) == expected` flakes on representation noise | `abs(a - b) < 1e-9` or `math.isclose` |
| Validating only at training time | Serving receives tonight's unvalidated feed | The SAME gate runs at ingestion AND before scoring |
| Launch-day thresholds forever | The 0.88 recall of March becomes the permanent bar | Re-derive gates from rolling baselines (Lesson 06) |
| Tests that hit live networks/files | Suite crawls, fails randomly, gets ignored | Seeded synthetic fixtures; network is a mocking concern |
| Testing only the happy path | One weird row takes down the nightly run | Corrupt a copy on purpose: nulls, ranges, strangers |
| Behaviour tests without data tests | Great model, fed garbage, confidently wrong | Layers 2 and 3 ship together (Section 1) |

## 💡 Best Practices & Pro Tips

- **Fail loud, fail early, fail at the door.** A batch rejected at ingestion costs minutes;
  the same poison discovered after training costs a week of forensics.
- **Keep fixtures tiny and deterministic** — ten seeded rows say more than a 2 GB sample,
  and they diff cleanly in review.
- **Name tests after the property, not the function**: `test_more_hours_never_lower_score`
  documents the contract; `test_model_3` documents nothing.
- **Every expectation gets a reason in a comment.** "Null share < 5% because the vendor
  guarantees it" survives reorgs; a naked number invites deletion.
- **AI-engineering relevance:** LLM apps translate one-to-one — golden eval sets are
  fixtures, refusal-of-harm checks are behaviour tests, and "answer stayed on-topic" is a
  directional expectation you can grade.
- **Track the gate, not just the metric**: when a gate blocks a release, log it (Lesson 03);
  blocked deploys are experiments too.

## 📌 Summary

| Tool / Pattern | What it does | Example |
|---|---|---|
| pytest discovery | Collects `test_*` functions, bare asserts | `pytest tests/ -q` |
| `-m pytest` from Python | Suites callable from pipelines/notebooks | `subprocess.run([sys.executable, "-m", "pytest", ...])` |
| `validate_batch(df, rules)` | Schema + range + category gate at ingestion | stops corrupt batches cold |
| Statistical expectations | Nulls, bands, uniqueness, label balance | `student_id.duplicated().any()` |
| Metric gate | Promotion needs explicit quality | `assert r_squared >= 0.90` |
| Direction / invariance tests | Pin the model's shape, not its number | more hours never lower the score |

Key takeaways:

- Three layers — code, data, behaviour — and layer 1 alone defends none of the interesting failures.
- Data gates belong at ingestion time, before training and before serving.
- Behavioural contracts (direction, invariance) catch bugs that aggregate metrics hide.
- Thresholds are living policy: derive them from baselines, revisit them on purpose.

## 🔗 Next Lesson

Up next: **[06_CI_CD_And_Monitoring](../06_CI_CD_And_Monitoring/notes.ipynb)** — wiring these gates into an automated delivery pipeline, choosing rollout patterns that limit blast radius, and catching drift with PSI before users do.